## 2026_Winter_DL_week5_Homework
CNN 실습 예제코드입니다!

이번 5주차 숙제는 총 2가지로 구성되어 있습니다.
- 논문 리뷰 - ResNet : https://arxiv.org/abs/1512.03385
    - 다음 논문을 읽고 팀별로 논문 스터디 및 정리를 준비해주시면 됩니다! (notion 이나 ppt 로 논문 정리 자료 제작 후에 팀별로 git에 업로드 해주세요 :D) 

- 코드 과제 (아래 두 가지 중 택1!)
    - 1) (CNN 을 처음 접해보시거나 기초를 더 다지고 싶은 분들께 추천)
        - 아래 실습코드를 그대로 참고하셔도 좋고, 새롭게 짜셔도 좋습니다. 데이터셋은 CIFAR10이 아닌 벤치마크(MNIST, FFHQ, ImageNet)이나 다른 데이터셋을 사용하되, 학습 시간 및 리소스를 적절히 사용할 수 있는 화질을 사용하는 것을 추천 드립니다. 또한, 앞서 배운 regularization, initialization, optimizer 등등 기법을 추가해보시거나, layer를 변형하는 시도를 추가하여 결과를 분석해주세요.
        - example : Earlystopping 추가, Dropoutlayer 추가, batch nomalization 추가, stride 및 padding 변형, Conv layer 추가 및 삭제 등등
    - 2) (이미 CNN 지식이 있는 분들) 
        - ResNet 을 논문 "**만**" 읽고 핵심 모듈 코드를 직접 짜보세요! 코드를 보지 않고 conv block이나 layer들을 논문 기반으로 짜보시면 됩니다.(임교수님께서 추천해주신 공부법입니다 ㅎㅎ)

In [3]:
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [14]:
#cell_1
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.adam import Adam
from torch.utils.data.dataloader import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from tqdm import tqdm

In [15]:
#cell_2
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(
    root='./data/',
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root='./data/',
    train=False,
    download=True,
    transform=transform
)

classes = train_data.classes

In [ ]:
#cell_3
from torch.utils.data import Dataset

class CustomCIFAR10(Dataset):
    def __init__(self, root, train=True, download=False):
        # self.base_dataset = dataset.load_dataset(path = '')
        self.base_dataset = datasets.cifar.CIFAR10(
            root=root,
            train=train,
            download=download
        )

        self.train = train

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.base_dataset.data)

    def __getitem__(self, idx):
        image = self.base_dataset.data[idx]  # numpy array (H, W, C)
        label = self.base_dataset.targets[idx]

        if self.train:
            image = self.transform(image)

        return image, label

# 인스턴스 생성
train_data = CustomCIFAR10(root='./data/', train=True, download=True)
test_data = CustomCIFAR10(root='./data/', train=False, download=True)

6.8%


KeyboardInterrupt: 

## Modeling

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            hidden_dim,
            kernel_size=5,
            padding=2
        )

        self.conv2 = nn.Conv2d(
            hidden_dim,
            out_channels,
            kernel_size=5,
            padding=2
        )

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        return x

In [17]:
#cell_5
class CNN(nn.Module):
    def __init__(self, num_classes): # 클래스의 갯수
        super(CNN, self).__init__()
        
        # CNN 블럭 정의
        self.block1 = BasicBlock(in_channels=1, out_channels=32, hidden_dim=16)
        self.block2 = BasicBlock(in_channels=32, out_channels=128, hidden_dim=64)
        self.block3 = BasicBlock(in_channels=128, out_channels=256, hidden_dim=128)
        
        # classification을 위한 FC 정의
        self.fc1 = nn.Linear(in_features=256*3*3, out_features=2048)
        self.fc2 = nn.Linear(in_features=2048, out_features=256)
        self.fc3 = nn.Linear(in_features=256, out_features=num_classes)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x) # 출력모양: (-1, 256, 4, 4)
        x = torch.flatten(x, start_dim=1) # 2차원의 feature map을 1차원으로
        
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        
        return x

In [ ]:
#cell_6
from torch.utils.data.dataloader import DataLoader

from torch.optim.adam import Adam


# 데이터로더 정의
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# 학습을 진행할 프로세서 설정
device = "cuda" if torch.cuda.is_available() else "cpu"


print(f"device 정보: {device}")

# CNN 모델 정의
model = CNN(num_classes=10)

# 모델을 device로 보냄
model.to(device)

device 정보: cpu


CNN(
  (block1): BasicBlock(
    (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU()
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): BasicBlock(
    (conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU()
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): BasicBlock(
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (relu): ReLU()
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=2304, out_features=2048, bias=True)
  (fc2): Linear(in_features=2048, out_features=256,

In [19]:
#cell_7
# Learning rate
lr = 1e-3

# Optimizer
optim = Adam(model.parameters(), lr=lr)

# Train loop
for epoch in range(10):
    for data, label in tqdm(train_loader): # read data
        optim.zero_grad() # Iniitlaization
        
        preds = model(data.to(device)) # prediction
        
        # Loss Function
        loss = nn.CrossEntropyLoss()(preds, label.to(device))
        loss.backward()
        optim.step()
    
    if epoch==0 or epoch%10==9: # 10번마다 손실 출력
        print(f"epoch{epoch+1} loss:{loss.item()}")
        
# 모델 저장
torch.save(model.state_dict(), "kernel5.pth")

100%|██████████| 938/938 [03:10<00:00,  4.93it/s]


epoch1 loss:0.37411534786224365


100%|██████████| 938/938 [01:43<00:00,  9.10it/s]


epoch10 loss:0.0384310744702816


In [25]:
#cell_8
model.load_state_dict(torch.load("baseline.pth", map_location=device))

num_corr = 0

with torch.no_grad():
    for data, label in tqdm(test_loader):
        output = model(data.to(device))
        preds = output.data.max(1)[1]
        corr = preds.eq(label.to(device).data).sum().item()
        num_corr += corr

    print(f"Accuracy:{num_corr/len(test_data)}")

100%|██████████| 157/157 [00:08<00:00, 19.61it/s]

Accuracy:0.9226


## 실험 결과 비교

베이스라인 모델에서는 `cell_4`의 `BasicBlock`에서 두 개의 convolution layer를 모두 `kernel_size=3`, `padding=1`로 설정하고, 이후 `MaxPool2d(kernel_size=2, stride=2)`를 적용하였다. 학습은 `cell_7`에서 learning rate를 `1e-3`으로 설정하고 Adam optimizer를 사용하여 총 10 epoch 동안 진행하였다. 베이스라인의 최종 테스트 정확도는 0.9226으로 약 92.26%였으며, 전체 실행에는 약 39분 28초가 소요되었다. 학습 loss는 1 epoch에서 약 0.3741, 10 epoch에서 약 0.0384까지 감소하였다.

실험 2에서는 convolution filter가 한 번에 더 넓은 영역을 볼 수 있도록 `cell_4`의 `BasicBlock`에서 convolution layer의 kernel size를 키웠다. 베이스라인에서는 `self.conv1`과 `self.conv2` 모두 `kernel_size=3`, `padding=1`을 사용했지만, 실험 2에서는 이를 `kernel_size=5`, `padding=2`로 변경하였다. 즉 수정한 부분은 `cell_4`에서 `conv1`, `conv2`를 정의하는 두 부분이며, pooling 방식이나 optimizer, learning rate, epoch 수 등 다른 조건은 동일하게 유지하였다. padding도 1에서 2로 함께 변경한 이유는 kernel size를 5로 늘리더라도 convolution 전후의 feature map 크기는 유지하기 위해서이다. 이를 통해 단순히 kernel이 바라보는 범위를 넓혔을 때 성능이 어떻게 변하는지 비교하였다.

실험 2의 최종 테스트 정확도는 0.9254로 약 92.54%였으며, 세 결과 중 가장 높은 정확도를 기록하였다. 베이스라인의 92.26%와 비교하면 약 0.28%p 향상되었다. 각 epoch에는 약 17~19초가 소요되었고, 10 epoch 전체 학습에는 약 3분 정도가 걸렸다. 따라서 kernel size를 3에서 5로 확장한 경우 이번 실험에서는 성능이 소폭 향상되는 결과를 확인하였다.

실험 3에서는 convolution layer는 다시 베이스라인과 동일하게 `kernel_size=3`, `padding=1`로 두고, pooling 방식만 변경하였다. 구체적으로 `cell_4`의 `self.pool = nn.MaxPool2d(kernel_size=2, stride=2)`에서 `stride=2`를 `stride=1`로 변경하였다. kernel size는 2로 그대로 유지하고 pooling이 이동하는 간격만 줄인 것이다. stride가 작아지면 pooling 과정에서 feature map의 크기가 더 천천히 감소하므로 공간 정보를 더 많이 유지할 수 있지만, 이후 layer에서 처리해야 하는 연산량도 증가한다.

실험 3에서는 한 epoch당 약 1분이 소요되어 10 epoch 학습에 약 10분이 필요하였다. 최종 테스트 정확도는 0.9227로 약 92.27%였으며, 베이스라인의 92.26%와 거의 동일하였다. 즉 pooling stride를 2에서 1로 줄였을 때 공간 정보를 더 많이 유지할 수는 있었지만, 이번 실험에서는 정확도 향상으로 이어지지 않았고 학습 시간만 증가하였다.

세 실험을 비교하면 베이스라인은 92.26%, 실험 2는 92.54%, 실험 3은 92.27%의 정확도를 기록하였다. 실험 2에서 `cell_4`의 convolution kernel을 `3×3`에서 `5×5`로 확장했을 때 가장 높은 정확도가 나타났으며, 실험 3에서 MaxPooling의 stride를 2에서 1로 줄였을 때는 베이스라인과 거의 동일한 정확도를 보였다. 따라서 이번 실험에서는 pooling을 더 촘촘하게 수행하는 것보다 convolution filter가 바라보는 영역을 넓히는 것이 성능 개선에 조금 더 효과적이었다.

실험 1도 수행을 시도하였으나 학습 시간이 지나치게 오래 소요되어 전체 결과를 끝까지 확인하기 어려웠기 때문에, 이번 비교에서는 우선 완료된 실험 2와 실험 3을 중심으로 정리하였다.ㅠㅠ